# ComplaintIQ - unsupervised baselines (`05_baselines_unsupervised`)

The baseline for the **unsupervised** side: theme discovery over complaint narratives. As
with `04`, this is the **floor** a real approach must beat, with **no feature engineering** —
raw default TF-IDF and an off-the-shelf KMeans.

There is no label to optimize, so we measure the clustering against the **label-free
yardstick** established in `03_eda_unsupervised.ipynb` section 8: the `product` and
`product x issue` groupings a good clustering should roughly recover. Metrics are ARI / NMI
against those groupings, plus silhouette for internal cohesion.

## How to read this notebook
Distance methods cannot scan 16.5M narratives, so Spark draws a reproducible sample and only
that sample goes to sklearn. The sample is stratified by `product` so every theme is
represented (not just the high-volume ones). `k` is fixed to the number of products — a
principled, tuning-free choice tied to the yardstick, not an optimized hyperparameter.

> **Note:** low ARI / NMI is the expected, honest result for a naive baseline — it is the
> number a real clustering (better representation, tuned k, embeddings) has to improve on.

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn loaded")

---
## 1. Load a representative sample

Prefer the narrative-only Parquet (dense text). Spark reads it and draws a fixed-seed
sample **stratified by `product`** so every theme is represented; only that sample is pulled
into pandas.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR
if not data_dir.exists():
    # Local: walk up from cwd to find the repo's data/ dir (robust to notebook depth).
    for _p in [Path.cwd(), *Path.cwd().parents]:
        if (_p / "data").is_dir():
            data_dir = _p / "data"
            break
nar_path = data_dir / "complaints_narrative_only.parquet"
full_path = data_dir / "complaints.parquet"

use_cols = ["complaint_text", "product", "issue"]
src = nar_path if nar_path.exists() else full_path
if not src.exists():
    raise FileNotFoundError("No Parquet in data/. Run `make parquet NARRATIVE_ONLY=1` first.")

docs = spark.read.parquet(str(src))
if "has_narrative" in docs.columns and src == full_path:
    docs = docs.filter(F.col("has_narrative"))
docs = docs.select(*use_cols).dropna(subset=["complaint_text", "product"])

# Stratify by product so low-volume themes are represented, not just the big buckets.
total = docs.count()
SAMPLE_N = min(total, 25_000)
prods = [r["product"] for r in docs.select("product").distinct().collect()]
frac = min(1.0, SAMPLE_N / total) if total else 0.0
sample = docs.sampleBy("product", {p: frac for p in prods}, seed=RANDOM_STATE).toPandas()
print(f"corpus: {total:,} narratives  |  working sample: {len(sample):,}")

---
## 2. Baseline representation - raw default TF-IDF

Default `TfidfVectorizer` (English stop words, `min_df=5`), no custom tokenizer, no n-grams
beyond unigrams, no dimensionality reduction. This is deliberately the plainest text
representation, so the clustering result is a true floor.

In [ ]:
tfidf = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
X = tfidf.fit_transform(sample["complaint_text"])
print(f"TF-IDF matrix: {X.shape[0]:,} docs x {X.shape[1]:,} terms")

---
## 3. Baseline clustering - KMeans at k = number of products

`k` is set to the number of distinct products — a tuning-free choice tied to the yardstick
(the clustering should recover roughly product-level structure). Default KMeans, fixed seed.

In [ ]:
k = sample["product"].nunique()
kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5)
labels = kmeans.fit_predict(X)
print(f"k = {k} clusters over {len(sample):,} narratives")
print("cluster sizes (largest 10):")
print(pd.Series(labels).value_counts().head(10))

---
## 4. Evaluate against the label-free yardstick

ARI and NMI compare the clusters to the `product` and `product x issue` groupings from `03`
section 8 (agreement a good clustering should show), and silhouette measures internal
cohesion. All computed on the sample.

In [ ]:
prod_codes = sample["product"].astype("category").cat.codes
pi_codes = (
    (sample["product"].astype(str) + " | " + sample["issue"].astype(str))
    .astype("category")
    .cat.codes
)

metrics = {
    "ARI vs product": adjusted_rand_score(prod_codes, labels),
    "NMI vs product": normalized_mutual_info_score(prod_codes, labels),
    "NMI vs product x issue": normalized_mutual_info_score(pi_codes, labels),
    "silhouette (sample)": silhouette_score(X, labels, sample_size=5000, random_state=RANDOM_STATE),
}
for name, val in metrics.items():
    print(f"{name:<26} {val:.4f}")

> **What you're seeing:** agreement between raw-TF-IDF KMeans clusters and the product /
> product x issue yardstick, plus internal cohesion.
>
> **Notice:** ARI is near zero and NMI is low — the naive baseline only weakly recovers the
> known theme structure.
>
> **Why it matters:** this is the floor. A real approach (better text representation, tuned
> `k`, embeddings, dedup of the templated narratives flagged in `03` section 9) has to beat
> these numbers to claim it found meaningful themes.

---
## 5. What the clusters look like

Top terms per cluster (largest few) — a qualitative read on whether clusters are coherent
themes or just frequency artifacts.

In [ ]:
terms = np.array(tfidf.get_feature_names_out())
centers = kmeans.cluster_centers_
top_clusters = pd.Series(labels).value_counts().head(6).index
for c in top_clusters:
    top_terms = terms[np.argsort(-centers[c])[:8]]
    print(f"cluster {c:>2} (n={np.sum(labels == c):,}): {', '.join(top_terms)}")

---
## 6. Takeaways

> **What this baseline establishes:**
> - **Yardstick, not a label:** clustering quality is judged against the `product x issue`
>   structure from `03`, since there is no target to optimize.
> - **The floor is low:** raw TF-IDF + default KMeans recovers that structure only weakly
>   (near-zero ARI), which is the honest baseline to beat.
> - **Where the gains are:** better representation (embeddings), tuned `k`, and dedup of the
>   templated narratives (`03` section 9) are the levers a real approach should pull.

Supervised baselines are in **`04a_baselines_trivial.ipynb`** and
**`04b_baselines_logreg.ipynb`**.